### VIF-test to test multicollinearity (~redundancy)

In [1]:
# pip install statsmodels
from statsmodels.stats.outliers_influence import variance_inflation_factor 
import pandas as pd

pd.set_option('display.float_format', lambda x: '%.1f' % x) 

import warnings
warnings.filterwarnings("ignore")

# load the data
df = pd.read_csv("../heart.csv")


# some missing values in the dataset, drop for VIF-test
df = df.dropna()



## Handling the categorical variables

#### Type 1: Boolean categories

In [2]:
# this just converts the value of column to 0 or 1
# factorize in pandas works too, but only one column at a time
from sklearn.preprocessing import LabelEncoder
variables = ['ExerciseAngina']
encoder = LabelEncoder()
df[variables] = df[variables].apply(encoder.fit_transform)

#### Type 2: Ordinal categories

In [3]:
cut_mapper = {'Down': 1, 'Flat': 2,'Up': 3}
df['ST_Slope'] = df['ST_Slope'].map(cut_mapper)

#### Type 3: Nominal categories

In [4]:
# this makes multiple columns with the variable (Separate for yes/no)
from sklearn.preprocessing import OneHotEncoder
variables = ['Sex', 'ChestPainType', 'RestingECG']

# use encoder
encoder = OneHotEncoder(sparse_output=False).set_output(transform="pandas")
one_hot_encoded = encoder.fit_transform(df[variables]).astype(int)
df = pd.concat([df,one_hot_encoded],axis=1).drop(columns=variables)

In [5]:
df = df.drop(["Sex_F", "ChestPainType_TA", "RestingECG_LVH"], axis=1)

In [6]:
# perform X/y -split
# if you  have more than one independent variable, list them all here
# leave out the target variable! (dependent variable)

# this is a nice and common trick => everything EXCEPT target variable => support variable
X = df.drop("HeartDisease", axis=1)

# have only the target variable here (dependent variable)
y = df["HeartDisease"]



# VIF dataframe 
# VIF = Variance Inflation Factor
vif_data = pd.DataFrame() 
vif_data["feature"] = X.columns 
  
# calculating VIF for each feature 
vif_data["VIF"] = [variance_inflation_factor(X.values, i) 
                          for i in range(len(X.columns))] 

# variables with high VIF-value 
# can mean multlicollinearity (variables providing same linear
# relationships in the data, potentially confusing the ML algorithm
# this might be good info when deciding if some variable needs to be removed
vif_data.sort_values(by="VIF", ascending=False)

,feature,VIF
9,ChestPainType_ASY,5.8
11,ChestPainType_NAP,4.3
10,ChestPainType_ATA,4.1
12,RestingECG_Normal,1.7
13,RestingECG_ST,1.7
7,ST_Slope,1.6
5,ExerciseAngina,1.6
4,MaxHR,1.5
6,Oldpeak,1.5
0,Age,1.4
